# Preference training

standard takes preference_input and model_name (normally an SFT adapter). It splits by question ID and exposes the trainer's loss_type, beta and declared CLI options.

Staged presets require base_model, initial_adapter, staged_root and dev_source_input. Set model_preset to qwen25_05b or qwen25_3b. The staged directory must contain all three stage files, even if a selected stage is empty. stage1/two_stage/three_stage set the stopping point; stage_settings overrides per-stage hyperparameters.

This notebook is preconfigured for the current C3>C1 experiment. It deduplicates and combines the 0.5B and 3B C3>C1 pair banks, then creates two independent stage1 plans that use that same combined dataset. stop_after_stage=concept_learning prevents C3>C2 and C2>C1 pairs from entering either run. The shared 160-question dev split is used only for checkpoint selection.

**Staged presets default to smoke_test=True.** After checking a smoke run, set False for full training. stage2_retention needs a completed Stage 1 adapter plus tie-aware data; it skips concept learning but may use its pairs as retention anchors. cal_dpo, apo_zero and adaptive_nll select other objectives.

softmax/representative require question_classes_with_reference_jsonl, not ordinary pairs. error_aware takes accepted C1/gold rationales and Stage 1 pairs; it prepares data unless its separate train=True parameter is set. Every run is fresh; checkpoints are not silently resumed.

Execution is **off by default**. Preview needs only Python and the repository. Select a kernel with the project's runtime dependencies before executing. Each method uses that kernel's Python in a separate worker process.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


## Choose a preset

- **standard**: standard_dpo
- **stage1**: staged_dpo
- **two_stage**: staged_dpo
- **three_stage**: staged_dpo
- **stage2_retention**: staged_dpo
- **cal_dpo**: staged_dpo
- **apo_zero**: staged_dpo
- **adaptive_nll**: staged_dpo
- **softmax**: orbit_dpo
- **representative**: orbit_dpo
- **error_aware**: error_aware_dpo

Edit the configuration, then rerun the remaining cells. OVERRIDES uses the option names printed by describe(), with underscores. For CLI flags, True includes the named flag and False omits it. Missing inputs block execution.


In [ ]:
CATEGORY = 'dpo'
PRESET = 'two_stage'
OVERRIDES = {
    # Set input/model paths here using the fields printed below.
}
RUN = True
ALLOW_API = False

# Optional sequential comparisons: shared OVERRIDES plus per-variant changes.
VARIANTS = [{"label": "default", "overrides": {}}]

# Current experiment: train both SFT models on the same combined C3>C1 pairs.
# RUN remains False so the cell first creates two reviewable plans. Set it to
# True and rerun this cell plus the cells below to launch the full runs.
from src.notebook_workflows.dpo_inputs import build_c3_c1_union

PRESET = "stage1"
SOURCE_STAGED_ROOTS = [
    PROJECT_ROOT
    / "Artifacts/notebook_runs/pairs/judge_candidates"
    / "20260929-030110-87693ae1/judgments/staged_curriculum_pairs",
    PROJECT_ROOT
    / "Artifacts/notebook_runs/pairs/judge_candidates"
    / "20260929-030110-b9ec9c29/judgments/staged_curriculum_pairs",
]
COMBINED_STAGED_ROOT = (
    PROJECT_ROOT
    / "Artifacts/notebook_inputs/dpo"
    / "c3_over_c1_combined_qwen25_05b_3b"
)
COMBINED_SUMMARY = build_c3_c1_union(
    SOURCE_STAGED_ROOTS,
    COMBINED_STAGED_ROOT,
    source_labels=["qwen25_05b_bank", "qwen25_3b_bank"],
)
print("Combined C3>C1 dataset:", json.dumps(COMBINED_SUMMARY, indent=2))

OVERRIDES = {
    "dev_source_input": str(
        PROJECT_ROOT
        / "data/BioASQ_factoid_sft_prepared"
        / "split_first_train90_dev10_supported_train_seed3407"
        / "dev_prepared.json"
    ),
    "objective": "dpo",
    "stop_after_stage": "concept_learning",
    "smoke_test": False,
    "max_length": 4096,
    "generated_eval_max_seq_length": 4096,
    "semantic_judge_enabled": False,
}
VARIANTS = [
    {
        "label": "qwen25_05b_c3_over_c1",
        "overrides": {
            "model_preset": "qwen25_05b",
            "base_model": "unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit",
            "initial_adapter": str(
                PROJECT_ROOT
                / "Artifacts/notebook_runs/sft/answer_sft"
                / "20260927-143545-67f7a338/adapter"
            ),
            "staged_root": str(COMBINED_STAGED_ROOT),
        },
    },
    {
        "label": "qwen25_3b_c3_over_c1",
        "overrides": {
            "model_preset": "qwen25_3b",
            "base_model": "unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit",
            "initial_adapter": str(
                PROJECT_ROOT
                / "Artifacts/notebook_runs/sft/answer_sft"
                / "20260927-143545-3fca1c3d/adapter"
            ),
            "staged_root": str(COMBINED_STAGED_ROOT),
        },
    },
]

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [ ]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [ ]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [ ]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])
